# 03 - Gold Load

Construcción de marts analíticos para consumo gerencial / Power BI.

Marts:
- `gold_daily_fintech_kpis`
- `gold_fraud_by_segment_channel`
- `gold_customer_risk_profile`


In [ ]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog_name", "fintech_lakehouse")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("gold_schema", "gold")

catalog_name = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")

spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{gold_schema}")

tx = spark.table(f"{catalog_name}.{silver_schema}.transactions_enriched_silver")

In [ ]:
gold_daily_fintech_kpis = (
    tx.groupBy("transaction_date")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount_pen").alias("total_amount_pen"),
        F.avg("amount_pen").alias("avg_ticket_pen"),
        F.sum(F.when(F.col("status") == "approved", 1).otherwise(0)).alias("approved_transactions"),
        F.sum(F.when(F.col("status") == "declined", 1).otherwise(0)).alias("declined_transactions"),
        F.sum("is_fraud_confirmed").alias("confirmed_fraud_transactions"),
        F.avg("risk_score").alias("avg_risk_score")
    )
    .withColumn("approval_rate", F.col("approved_transactions") / F.col("total_transactions"))
    .withColumn("fraud_rate", F.col("confirmed_fraud_transactions") / F.col("total_transactions"))
)

gold_fraud_by_segment_channel = (
    tx.groupBy("segment", "channel", "risk_bucket")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum("amount_pen").alias("total_amount_pen"),
        F.sum("is_fraud_confirmed").alias("confirmed_fraud_transactions"),
        F.avg("risk_score").alias("avg_risk_score"),
        F.max("risk_score").alias("max_risk_score")
    )
    .withColumn("fraud_rate", F.col("confirmed_fraud_transactions") / F.col("total_transactions"))
)

gold_customer_risk_profile = (
    tx.groupBy("customer_id", "segment", "district", "risk_level", "kyc_status")
    .agg(
        F.count("*").alias("transactions_count"),
        F.sum("amount_pen").alias("total_amount_pen"),
        F.avg("amount_pen").alias("avg_ticket_pen"),
        F.sum("is_fraud_confirmed").alias("confirmed_fraud_transactions"),
        F.max("risk_score").alias("max_risk_score"),
        F.avg("risk_score").alias("avg_risk_score")
    )
    .withColumn("fraud_rate", F.col("confirmed_fraud_transactions") / F.col("transactions_count"))
    .withColumn(
        "customer_risk_classification",
        F.when((F.col("max_risk_score") >= 80) | (F.col("fraud_rate") >= 0.20), "critical")
         .when((F.col("max_risk_score") >= 60) | (F.col("fraud_rate") >= 0.10), "high")
         .when(F.col("max_risk_score") >= 40, "medium")
         .otherwise("low")
    )
)

In [ ]:
gold_daily_fintech_kpis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{gold_schema}.gold_daily_fintech_kpis")
gold_fraud_by_segment_channel.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{gold_schema}.gold_fraud_by_segment_channel")
gold_customer_risk_profile.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.{gold_schema}.gold_customer_risk_profile")

display(gold_daily_fintech_kpis.orderBy(F.desc("transaction_date")).limit(20))
display(gold_fraud_by_segment_channel.orderBy(F.desc("fraud_rate")).limit(20))
display(gold_customer_risk_profile.orderBy(F.desc("max_risk_score")).limit(20))